# QuEstX Dataset Generator

This notebook generates a dataset of random quantum circuits with their corresponding hardware calibration data and simulated PST (Probability of Successful Trial) labels.

1. Dependency Installation

In [ ]:
# Install required libraries for quantum simulation
%pip install qiskit qiskit-aer

2. Core Simulation and Generation Logic

In [ ]:
import pickle
import random
import numpy as np
import os
import gc
from qiskit import QuantumCircuit, transpile

# Tenta importar qasm2 de forma segura
try:
    from qiskit import qasm2
except ImportError:
    qasm2 = None

from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    thermal_relaxation_error,
    depolarizing_error,
    ReadoutError
)

# --- CONFIGURATIONS & HYPERPARAMETERS ---
# These parameters define the D2 scale (up to 18 qubits for initial D2 tests)
NUM_SAMPLES = 15000
MIN_QUBITS = 15
MAX_QUBITS = 18
MIN_DEPTH = 20
MAX_DEPTH = 50
BATCH_SIZE = 25
OUTPUT_FILENAME = "dataset_D2"
SHOTS = 1024

# --- GATE SET DEFINITION ---
# Expanded gate set (13 types) as described in Experiment E2
GATES_1Q_NO_PARAM = ['x', 'sx', 'h', 'y', 'z', 's', 't']
GATES_1Q_PARAM = ['rz', 'rx', 'ry']
GATES_2Q = ['cx', 'cz', 'swap']

# Approximate gate durations in nanoseconds (ns) for thermal relaxation modeling
GATE_TIMES = {
    'rz': 0, 'sx': 35, 'x': 35, 'h': 35, 'y': 35, 'z': 0, 's': 0, 't': 0,
    'rx': 35, 'ry': 35,
    'cx': 300, 'cz': 300, 'swap': 600,
    'measure': 1000
}

# --- HELPER FUNCTIONS ---

def get_qasm_str(circuit):
    """Robustly exports the circuit to OpenQASM string format."""
    if qasm2 is not None:
        try:
            return qasm2.dumps(circuit)
        except:
            pass
    return circuit.qasm()

def get_expanded_random_circuit(n_qubits, depth):
    """Generates a random DAG-structured circuit with expanded gate diversity."""
    qc = QuantumCircuit(n_qubits)
    gate_pool = GATES_1Q_NO_PARAM + GATES_1Q_PARAM + GATES_2Q

    for _ in range(depth):
        gate_name = random.choice(gate_pool)

        # 2-Qubit Gates (cx, cz, swap)
        if gate_name in GATES_2Q:
            if n_qubits >= 2:
                q_ids = random.sample(range(n_qubits), 2)

                if gate_name == 'cx':
                    qc.cx(q_ids[0], q_ids[1])
                elif gate_name == 'cz':
                    qc.cz(q_ids[0], q_ids[1])
                elif gate_name == 'swap':
                    qc.swap(q_ids[0], q_ids[1])

        # Parameterized 1-Qubit Gates (rz, rx, ry)
        elif gate_name in GATES_1Q_PARAM:
            q_idx = random.randint(0, n_qubits - 1)
            angle = random.uniform(0, 2 * np.pi)
            # Usa getattr para chamar rx, ry ou rz dinamicamente
            getattr(qc, gate_name)(angle, q_idx)

        # Simple 1-Qubit Gates
        else:
            q_idx = random.randint(0, n_qubits - 1)
            getattr(qc, gate_name)(q_idx)

    return qc

def add_inverse_circuit(qc):
    """Applies the inverse-circuit protocol (Mirror Circuit) to calculate PST."""
    full_qc = qc.copy()
    inverse_qc = qc.inverse()
    full_qc = full_qc.compose(inverse_qc)
    full_qc.measure_all()
    return full_qc

def generate_calibration_data(n_qubits):
    """
    Simulates hardware noise parameters:
    - T1/T2 Relaxation times
    - Readout Errors
    - Gate Errors (Depolarizing + Thermal Relaxation)
    """
    noise_model = NoiseModel()
    props_dict = {'qubit': {}, 'gate': {}}
    t1_t2_ns = {}

    # Qubit Properties: T1 between 50us-200us; T2 limited by 2*T1
    for i in range(n_qubits):
        t1 = random.uniform(50, 200)
        t2 = random.uniform(20, min(2 * t1, 300))
        p_readout = random.uniform(0.01, 0.05)

        props_dict['qubit'][i] = {
            'T1': t1,
            'T2': t2,
            'prob_meas0_prep1': p_readout,
            'prob_meas1_prep0': p_readout
        }
        t1_t2_ns[i] = (t1 * 1000, t2 * 1000)

        ro_error = ReadoutError([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
        noise_model.add_readout_error(ro_error, [i])


    # Gate Error Models (1Q and 2Q)
    for i in range(n_qubits):
        t1, t2 = t1_t2_ns[i]
        props_dict['gate'][(i,)] = {}

        for gate in GATES_1Q_NO_PARAM + GATES_1Q_PARAM:
            gate_err = random.uniform(0.0001, 0.002)
            props_dict['gate'][(i,)][gate] = gate_err

            dep_error = depolarizing_error(gate_err, 1)
            time = GATE_TIMES.get(gate, 0)

            if time > 0:
                therm_error = thermal_relaxation_error(t1, t2, time)
                total_error = dep_error.compose(therm_error)
            else:
                total_error = dep_error

            noise_model.add_quantum_error(total_error, gate, [i])

    # 2-Qubit Gate Noise
    for i in range(n_qubits):
        for j in range(n_qubits):
            if i == j: continue

            if (i, j) not in props_dict['gate']:
                props_dict['gate'][(i, j)] = {}

            t1_a, t2_a = t1_t2_ns[i]
            t1_b, t2_b = t1_t2_ns[j]

            for gate in GATES_2Q:
                gate_err = random.uniform(0.005, 0.03)
                props_dict['gate'][(i, j)][gate] = gate_err

                dep_error = depolarizing_error(gate_err, 2)
                time = GATE_TIMES.get(gate, 0)

                if time > 0:
                    therm_a = thermal_relaxation_error(t1_a, t2_a, time)
                    therm_b = thermal_relaxation_error(t1_b, t2_b, time)
                    # Expandir erro térmico para o espaço de 2 qubits
                    therm_error = therm_a.expand(therm_b)
                    total_error = dep_error.compose(therm_error)
                else:
                    total_error = dep_error

                noise_model.add_quantum_error(total_error, gate, [i, j])

    return props_dict, noise_model

def calculate_pst(counts, n_qubits, shots):
    """Calculates the ratio of successful all-zero state trials (PST)."""
    target = '0' * n_qubits
    success = 0
    for k, v in counts.items():
        k_clean = k.replace('0x', '').replace(' ', '')
        if set(k_clean) == {'0'} or k_clean == '':
            success += v
    return success / shots


# --- MAIN EXECUTION LOOP ---
def main():
    os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
    sim = AerSimulator()
    full_dataset = []

    # Resume capability: loads existing data if the file exists
    if os.path.exists(OUTPUT_FILENAME):
        try:
            with open(OUTPUT_FILENAME, 'rb') as f:
                full_dataset = pickle.load(f)
                if isinstance(full_dataset, list):
                    print(f"Resuming: {len(full_dataset)} circuits loaded.")
                else:
                    print("Old format detected. Restarting list.")
                    full_dataset = []
        except Exception:
            print("Starting from scratch.")
            full_dataset = []

    generated_count = len(full_dataset)

    if generated_count >= NUM_SAMPLES:
        print("Dataset already complete!")
        return

    print(ff"Generating {NUM_SAMPLES - generated_count} remaining circuits...")

    for i in range(generated_count, NUM_SAMPLES):
        try:
            # 1. Circuit Construction
            n_q = random.randint(MIN_QUBITS, MAX_QUBITS)
            depth = random.randint(MIN_DEPTH, MAX_DEPTH)
            qc_orig = get_expanded_random_circuit(n_q, depth)
            qc_test = add_inverse_circuit(qc_orig)

            # 2. Calibration & Noise Setup
            props, noise_model = generate_calibration_data(n_q)

            # 3. Transpilation & Simulation
            my_basis_gates = GATES_1Q_NO_PARAM + GATES_1Q_PARAM + GATES_2Q
            qc_transpiled = transpile(qc_test, optimization_level=0, basis_gates=my_basis_gates)

            result = sim.run(qc_transpiled, noise_model=noise_model, shots=SHOTS).result()
            counts = result.get_counts()

            # 4. Result Aggregation
            pst = calculate_pst(counts, n_q, SHOTS)
            qasm = get_qasm_str(qc_orig)

            # Memory Management
            data_instance = [qasm, props, pst]
            full_dataset.append(data_instance)
            del qc_orig, qc_test, qc_transpiled, noise_model

        except Exception as e:
            print(f"Error at sample {i}: {e}")
            continue

        # Save progress and trigger garbage collection periodically
        if len(full_dataset) % BATCH_SIZE == 0:
            with open(OUTPUT_FILENAME, 'wb') as f:
                pickle.dump(full_dataset, f)
            print(f"Progresso salvo: {len(full_dataset)}/{NUM_SAMPLES}. PST recente: {pst:.4f}")
            gc.collect()

    # Final Save
    with open(OUTPUT_FILENAME, 'wb') as f:
        pickle.dump(full_dataset, f)

    print(f"Task complete! File saved: {OUTPUT_FILENAME}")

if __name__ == "__main__":
    main()

### Summary for Users

    Target: This script generates labeled data for high-qubit circuits (15–18 qubits) to test the scalability of Graph Transformer models.

    Data Format: Each entry in the saved pickle file is a list: [OpenQASM_String, Calibration_Dict, PST_Value].

    Reproducibility: The script simulates thermal relaxation (T1​,T2​) and depolarizing noise, mimicking real NISQ device characteristics.